**Fragment ranking**

Requirements: 
1. Fragment library, in sdf format.
2. Structures of targets with fragments bound, in pdb format. Structures should not have any atoms or residues missing.
3. JSON dictionary of pdb file with associated fragment (in SMILES string), for example : {"mArh-x1018.pdb": "Clc1ccc2nnnn2n1",...}

create sdf with just active site fragments from A71EV2A fragalysis data

In [1]:
import os
import shutil
import json
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import PandasTools
from rdkit.Chem import SDWriter
import oddt
from oddt.toolkits import rdk, ob
import openbabel
from sklearn.utils.deprecation import deprecated

In [ ]:
# csv of all A71EV2A structures
EV2A_metadata_csv = pd.read_csv("EV2A_metadata.csv")
# rename frist columnn from 'Code' to 'Name'
EV2A_metadata_csv_names = EV2A_metadata_csv.rename(columns={EV2A_metadata_csv.columns[0]: 'Name'})
EV2A_metadata_csv_names.head(1)

,Name,Long code,Experiment code,Compound code,Smiles,Centroid res,Downloaded,ConformerSites upload name,CanonSites upload name,CrystalformSites upload name,...,[Other] upload_1 2024-12-06,[Other] upload_1 2025-02-12,[Other] upload_2 2025-02-28,[Other] upload_6 2025-09-15,[Other] upload_7 2025-09-29,Main status,GOOD count,MEDIOCRE count,BAD count,RefinementResolution
0,A0450a,A71EV2A-x0450_A_201_v1,A71EV2A-x0450,Z100642432,CN(C)C(=O)c1ccc(F)cc1Br,A71EV2A-x0911/A/133/A_v1,True,1b - A71EV2A-x0911/A/147,1 - A71EV2A-x0911/A/147/1,F2b - A71EV2A-x0450/A/201/1,...,True,True,False,False,False,NaN,NaN,NaN,NaN,1.61


In [27]:
# select only Name, Smiles and Active site fragment columns for those where Active site fragment = True
active_site_frags = EV2A_metadata_csv_names[EV2A_metadata_csv_names['[Other] Active site fragment'] == True][['Name', 'Smiles', '[Other] Active site fragment']]
active_site_frags.head(1)

,Name,Smiles,[Other] Active site fragment
0,A0450a,CN(C)C(=O)c1ccc(F)cc1Br,True


In [28]:
# add RDKit mol to active_site_frags data frame
PandasTools.AddMoleculeColumnToFrame(active_site_frags, smilesCol='Smiles')
active_site_frags.head(1)

,Name,Smiles,[Other] Active site fragment,ROMol
0,A0450a,CN(C)C(=O)c1ccc(F)cc1Br,True,<rdkit.Chem.rdchem.Mol object at 0x30b2e2ab0>


In [ ]:
# optional visualisation 
# PandasTools.FrameToGridImage(active_site_frags.head(8), molsPerRow=4)

In [83]:
# create sdf of just the active site fragments, making sure name is included (needs to match pdb file names)
writer = SDWriter('active_site_frags.sdf')

for _, row in active_site_frags.iterrows():
    mol = row['ROMol']  # the RDKit molecule
    if mol is None:
        continue  # skip missing molecules

    # Set the molecule name to match the PDB folder
    mol.SetProp("_Name", row['Name'])

    # Optional: you can add other properties, e.g. SMILES
    mol.SetProp("SMILES", row['Smiles'])

    writer.write(mol)

writer.close()
print(f"Written {len(active_site_frags)} molecules to active_site_frags.sdf")


Written 44 molecules to active_site_frags.sdf


create folder with ligand/protein pdb structures for just the active site fragments

In [84]:
source_base = "/Users/s2605092/Data/EV2A_HIPPO/A71EV2A/aligned_files"
target_folder = "/Users/s2605092/Data/Fragment-ranking/Active_site_frag_pdbs"

frag_to_pdb = {}
for _, row in active_site_frags.iterrows():
    name = row['Name']
    if name in frag_to_pdb:
        print(f"Skipping duplicate: {name}")
        continue
    pdb_dir = os.path.join(target_folder, name)
    os.makedirs(pdb_dir, exist_ok=True)
    pdb_src = os.path.join(source_base, name, f"{name}.pdb")

    # print(pdb_src)
    if os.path.exists(pdb_src):
        pdb_dst = os.path.join(pdb_dir, f"{name}.pdb")
        if not os.path.exists(pdb_dst):
            shutil.copy2(pdb_src, pdb_dst)
            # print(f"Copied 1 PDB file to {pdb_dir}")
        # else:
            # print(f"File already exists, skipping copy: {pdb_dst}")
        frag_to_pdb[name] = {'pdb_file': pdb_dst, 'smiles': row['Smiles']}       
    else:
        print(f"Warning: PDB file for {name} not found at {pdb_src}")
        


create json file linking smiles to pdb file

In [85]:
with open(os.path.join("/Users/s2605092/Data/Fragment-ranking", "fragment_to_pdb.json"), "w") as f:
    json.dump(frag_to_pdb, f, indent=2)


test!

In [2]:
!python "/Users/s2605092/Software/fragment-ranking/src/generate_IFPs.py" \
  -IFP residue \
  -sdf "/Users/s2605092/Data/Fragment-ranking/active_site_frags.sdf" \
  -exps "/Users/s2605092/Data/Fragment-ranking/fragment_to_pdb.json" \
  -pdbs "/Users/s2605092/Data/Fragment-ranking/Active_site_frag_pdbs"

# odddt doesnt seem to be seeing rdkit/openbabel? or not able to read pdbs correctly


Total unique compounds in library: 41
A0365a
A0365a error
no mode for empty data
A0487a
A0487a error
no mode for empty data
A0237a
A0237a error
no mode for empty data
A1209a
A1209a error
no mode for empty data
A0836b
A0836b error
no mode for empty data
A1148a
A1148a error
no mode for empty data
A0351a
A0351a error
no mode for empty data
A0450a
A0450a error
no mode for empty data
A1019a
A1019a error
no mode for empty data
A0501a
A0501a error
no mode for empty data
A0739a
A0739a error
no mode for empty data
A0207a
A0207a error
no mode for empty data
A1180a
A1180a error
no mode for empty data
A1169a
A1169a error
no mode for empty data
A0812a
A0812a error
no mode for empty data
A0926a
A0926a error
no mode for empty data
A0375b
A0375b error
no mode for empty data
A1080a
A1080a error
no mode for empty data
A1255a
A1255a error
no mode for empty data
A0554a
A0554a error
no mode for empty data
A0514a
A0514a error
no mode for empty data
A0911a
A0911a error
no mode for empty data
A0446a
A0446a er